In [26]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import interpolate
import tqdm
import re
import os
import glob
import spectres
from astropy import units as u

%matplotlib inline

In [51]:
SPS_HOME = os.path.abspath(os.path.join(os.getcwd(), '..'))
# SPS_HOME = os.getenv('SPS_HOME')
# SPS_HOME = SPS_HOME.replace('fsps', 'fsps_dev')  # -> I call my development folder for fsps 'fsps_dev' to keep it separate from my working fsps installation

# choose one metallicity value for this run
logzi = -2.
zstr = ('z+' if logzi >= 0.0 else 'z-') + '{:.2f}'.format(round(logzi, 1)).replace('.','').replace('-','')

print(SPS_HOME)
print(zstr)

/Users/mreefe/Dropbox/Astrophysics/fsps_dev
z-200


In [52]:
# Define the teff, logg, and logz arrays that cover the grid of Brown+1996 models

teff_1 = np.arange(10000, 13000, 500)
teff_2 = np.arange(13000, 20000, 1000)
teff_3 = np.arange(20000, 30000, 2000)
teff_4 = np.arange(30000, 50000, 2500)
teff_5 = np.arange(50000, 120000, 5000)
teff_6 = np.arange(120000, 200000, 20000)
teff_7 = np.arange(200000, 250000+25000, 25000)
teff = np.concatenate((teff_1, teff_2, teff_3, teff_4, teff_5, teff_6, teff_7))
logt = np.log10(teff)

logg = np.arange(2., 8.75, 0.25)

logz = np.array([-2., -1., 0.])

print(logt)
print(logg)
print(logz)

[4.         4.0211893  4.04139269 4.06069784 4.07918125 4.09691001
 4.11394335 4.14612804 4.17609126 4.20411998 4.23044892 4.25527251
 4.2787536  4.30103    4.34242268 4.38021124 4.41497335 4.44715803
 4.47712125 4.51188336 4.54406804 4.57403127 4.60205999 4.62838893
 4.65321251 4.67669361 4.69897    4.74036269 4.77815125 4.81291336
 4.84509804 4.87506126 4.90308999 4.92941893 4.95424251 4.97772361
 5.         5.0211893  5.04139269 5.06069784 5.07918125 5.14612804
 5.20411998 5.25527251 5.30103    5.35218252 5.39794001]
[2.   2.25 2.5  2.75 3.   3.25 3.5  3.75 4.   4.25 4.5  4.75 5.   5.25
 5.5  5.75 6.   6.25 6.5  6.75 7.   7.25 7.5  7.75 8.   8.25 8.5 ]
[-2. -1.  0.]


In [53]:
# Define the wavelength grid that we will resample onto
# -> even logarithmic spacing by ~5% of the current wavelength
w_1 = 90. * 1.05**np.arange(0, int(np.log(800/90)/np.log(1.05))) 
# -> finer sampling in the FUV;
w_2 = np.arange(800., 1800., 0.2)
w_3 = np.arange(1800., 9000., 20.)
# -> spacing by ~5% of the current wavelength
w_4 = 9000. * 1.05**np.arange(0, int(np.log(9.99e6/9000)/np.log(1.05))) 

wavelength = np.concatenate((w_1, w_2, w_3, w_4))
print('w_1 = ', len(w_1))
print('w_2 = ', len(w_2))
print('w_3 = ', len(w_3))
print('w_4 = ', len(w_4))
print('length = ', len(wavelength))
print(wavelength)

w_1 =  44
w_2 =  5000
w_3 =  360
w_4 =  143
length =  5547
[9.00000000e+01 9.45000000e+01 9.92250000e+01 ... 8.33190634e+06
 8.74850165e+06 9.18592674e+06]


In [54]:
# wavelength must be in angstroms!
def airtovac(wavelength):
    # see: https://www.astro.uu.se/valdwiki/Air-to-vacuum%20conversion
    s = 1e4 / wavelength
    n = 1 + 0.00008336624212083 + 0.02408926869968 / (130.1065924522 - s**2) + 0.0001599740894897 / (38.92568793293 - s**2)
    # do not alter wavelengths below 2000 angstroms 
    wh = np.where(wavelength < 2000.)[0]
    n[wh] = 1.0
    return wavelength * n

In [55]:

# allocate a buffer for all of the spectra at one metallicity
library_in = np.zeros((len(wavelength), len(logt), len(logg)))

folder  = f'/Users/mreefe/Dropbox/Astrophysics/stellar_templates/brownspec_resamp/{zstr}'
files = glob.glob(os.path.join(folder, '*'))

for fpath in tqdm.tqdm(files):

    # parse the file name to get the temp, logg, and logz
    fname = os.path.basename(fpath)
    m = re.search(r'^T(\d+)g(\d+)z([-+]\d+).spec$', fname)
    teff_v = int(m.group(1))
    logg_v = int(m.group(2))/100
    logz_v = int(m.group(3))/100

    # find the indices corresponding to these values in the array
    logt_i = np.where(teff == teff_v)[0][0]
    logg_i = np.where(logg == logg_v)[0][0]

    # read in the text files
    wave_i, flux_i = np.loadtxt(fpath,  unpack=True)

    # perform the flux-conserving resampling onto the output wavelength grid
    # flux_o = spectres.spectres(wavelength, wave_i, flux_i, fill=0., verbose=False)
    flux_o = np.interp(wavelength, wave_i, flux_i, left=0., right=0.)

    # insert it into the 3D array
    library_in[:, logt_i, logg_i] = flux_o

    # # plot the new and old spectrum to compare them
    # fig, ax = plt.subplots()
    # ax.plot(wave_i, flux_i)
    # ax.plot(wavelength, flux_o)
    # ax.set_xscale('log')
    # ax.set_yscale('log')
    # ax.set_xlabel('Wavelength (angstroms)')
    # ax.set_ylabel('Flambda')
    # # ax.set_xlim(900, 1800)
    # # ax.set_ylim(flux_i[(wave_i > 900) & (wave_i < 1800)].min()*0.98, flux_i[(wave_i > 900) & (wave_i < 1800)].max()*1.02)
    # plt.show()
    # plt.close()


  0%|          | 0/499 [00:00<?, ?it/s]

100%|██████████| 499/499 [00:07<00:00, 70.89it/s] 


In [56]:
# Convert the units
c_ang = 299792458e10

for i in range(len(logt)):
    for j in range(len(logg)):
        library_in[:,i,j] *= wavelength**2 / c_ang    # <= convert to erg/s/cm2/Hz
        library_in[:,i,j] *= 1/(4*np.pi)              # <= convert to Harvard flux
        # note: insofar as I can tell, FSPS stores its stellar libraries in flux moment or "Harvard flux" units, 
        #       which are off from physical flux units by a factor of 4pi.  See page 244-245 in 
        #       https://ads.harvard.edu/books/1989fsa..book/AbookC09.pdf for more info.
        #       Also see line 199 in getspec.f90 which does the conversion from these units into Lsun/Hz.


In [57]:
# Normalize to unity, to match the WMBasic grids
for i in range(len(logt)):
    for j in range(len(logg)):
        mask = np.isfinite(library_in[:,i,j])
        norm = np.trapz(library_in[:,i,j][mask]*c_ang/wavelength[mask]**2, wavelength[mask])
        if norm > 0:
            library_in[:,i,j] /= norm

In [58]:
# # Read in WMBasic templates for comparison
# wmb_logt = np.loadtxt(os.path.join(SPS_HOME, 'SPECTRA/Hot_spectra/WMBASIC.teff'))
# wmb_logg = np.array([3.5, 4., 4.5])
# wmb = np.loadtxt(os.path.join(SPS_HOME, 'SPECTRA/Hot_spectra/WMBASIC_z0.0140.spec'))
# w_wmb = wmb[:,0]
# wmb = wmb[:,1:]
# wmb = wmb.reshape(wmb.shape[0], len(wmb_logg), len(wmb_logt))

# template_folder = os.path.join(SPS_HOME, 'SPECTRA/Hot_spectra/Brown1996/template_plots')
# if not os.path.exists(template_folder):
#     os.makedirs(template_folder)


# jj = np.nanargmin(np.abs(wmb_logg[0] - logg))
# ii = np.nanargmin(np.abs(wmb_logt[1] - logg))

# win = (wavelength > 900) & (wavelength < 1800)
# window = (w_wmb > 900) & (w_wmb < 1800)
# scale = np.nanmedian(library_in[:,ii,jj][win]) / np.nanmedian(wmb[:,0,1][window])

# for j in range(len(wmb_logg)):
#     for i in range(len(wmb_logt)):
#         jj = np.nanargmin(np.abs(wmb_logg[j] - logg))
#         ii = np.nanargmin(np.abs(wmb_logt[i] - logt))
#         fig, ax = plt.subplots()
#         ax.plot(w_wmb, wmb[:,j,i], label='WMBASIC')
#         ax.plot(wavelength, library_in[:,ii,jj], label='TLUSTY')
#         ax.set_xscale('log')
#         ax.set_yscale('log')
#         ax.set_xlabel(r'Wavelength ($\mathring{\rm A}}$)')
#         ax.set_ylabel(r'Flux moment (erg s$^{-1}$ cm$^{-2}$ Hz$^{-1}$)')
#         # ax.set_xlim(900, 1800)
#         # ax.set_ylim(1e-22, 1e-14)
#         ax.set_title(f'logg={wmb_logg[j]}, teff={10**wmb_logt[i]}')
#         ax.legend()
#         plt.savefig(os.path.join(template_folder, f'{j}_{i}.pdf'), dpi=300, bbox_inches='tight')
#         plt.close()

In [59]:
library_in.shape

(5547, 47, 27)

In [60]:
# Save as a text file with the same format as the format 
zsun = 0.0134
z = 10**logzi * zsun
# need to flatten the array to 2D matching the shape of the WM-Basic arrays FSPS uses
# (flatten logt and logg axis, should be ordered as [t1_g1 t2_g1 t3_g1 ... t1_g2 t2_g2 t3_g2 ...])
library_out = library_in.reshape(library_in.shape[0], len(logg)*len(logt), order='F')
# append the wavelength column
library_out = np.concatenate((wavelength.reshape(len(wavelength), 1), library_out), axis=1)

np.savetxt(os.path.join(SPS_HOME, f'SPECTRA/Hot_spectra/Brown1996/Brown1996z{z:.4f}.spec'), library_out, fmt='%.4e', delimiter=' ')

In [61]:
# save other ancillary files
np.savetxt(os.path.join(SPS_HOME, 'SPECTRA/Hot_spectra/Brown1996/Brown1996.teff'), logt, fmt='%13.5f')
np.savetxt(os.path.join(SPS_HOME, 'SPECTRA/Hot_spectra/Brown1996/Brown1996.logg'), logg, fmt='%13.5f')
np.savetxt(os.path.join(SPS_HOME, 'SPECTRA/Hot_spectra/Brown1996/Brown1996_zlegend.dat'), 10**logz*zsun, fmt='%7.4f')